# Week 4: In-class warm-up problems

**Course:** AI for Aerospace & Mechanical Engineering (PhD), Block 0
**Format:** 4 problems, 5 to 10 minutes each, pen and paper first, then verify in Python.

> **Provenance:** new content. Pattern definitions rewritten from GeeksforGeeks ("Strategy Design
> Pattern", "Observer Design Pattern", "Factory Method - Python Design Patterns") and Gamma et al.,
> *Design Patterns* (1994); gradient descent and early stopping adapted from Goodfellow et al.,
> *Deep Learning* (2016), Sec. 4.3 and 7.8; fuel-flow model adapted from Anderson, *Aircraft
> Performance and Design*, Ch. 5.

## Problem 1 (10 min): one gradient-descent step by hand

<div dir="rtl">مسئله ۱: برای مدل خطی و سه نقطه‌ی داده، یک گام کاهش گرادیان را با دست انجام دهید و اثر نرخ یادگیری بزرگ را بررسی کنید.</div>

Model $\hat y = w_0 + w_1 x$, loss $L = \frac1n\sum_i(\hat y_i - y_i)^2$, data
$(x, y) = (1, 2), (2, 4), (3, 5)$, start $w = (0, 0)$.

a) Compute $L$ at the start.
b) Compute $\partial L/\partial w_0$ and $\partial L/\partial w_1$.
c) Take one step with $\eta = 0.1$. Give the new $w$ and the new loss.
d) Repeat c) with $\eta = 0.5$. What happened, and which callback from the lecture would catch it?

## Problem 2 (10 min): trace early stopping

<div dir="rtl">مسئله ۲: شمارنده‌ی صبر (patience) را برای دنباله‌ی خطای اعتبارسنجی دنبال کنید و بگویید آموزش در کدام دوره متوقف می‌شود و وزن‌های کدام دوره بازگردانده می‌شوند.</div>

`EarlyStopping(patience=2, min_delta=0.05, restore_best=True)` watches these validation losses:

| epoch | 0 | 1 | 2 | 3 | 4 | 5 | 6 |
|-------|---|---|---|---|---|---|---|
| val_loss | 1.00 | 0.80 | 0.78 | 0.70 | 0.69 | 0.68 | 0.50 |

An epoch counts as an improvement only if `val_loss < best - min_delta`.

a) Write `best` and `wait` after each epoch until training stops.
b) After which epoch does training stop, and whose weights does the model end with?
c) Repeat with `patience=3`. What changes?
d) In one sentence each: the engineering cost of a patience that is too small, and too large.

## Problem 3 (10 min): which strategy for the flight-test campaign?

<div dir="rtl">مسئله ۳: سه راهبرد (میانگین، نزدیک‌ترین همسایه و خط کمترین مربعات) را روی چهار پرواز آزمایشی آموزش دهید، خطای RMSE آزمون را حساب کنید و درباره‌ی برون‌یابی به جرم ۴۵ کیلوگرم بحث کنید.</div>

Four test flights at $V = 25$ m/s, sea level, 1 m/s climb, with different payloads:

| mass [kg] | 25 | 30 | 35 | 40 |
|-----------|----|----|----|----|
| fuel flow [g/h] | 364.0 | 408.2 | 456.4 | 508.4 |

Next week's flights: 32 kg (measured 427.0 g/h) and 45 kg (measured 564.3 g/h).

a) Predict both with `MeanBaseline`, `KNearestNeighbors(k=1)` and `LinearLeastSquares`
   (compute slope and intercept by hand).
b) Compute the test RMSE of each strategy. Which one would you deploy?
c) The 45 kg flight is outside the training range. Why does every strategy under-predict it?
   (Hint: how does induced power depend on mass?)
d) Which line of `RegressionExperiment` changes between the three runs?

## Problem 4 (5 to 10 min): name the pattern, predict the output

<div dir="rtl">مسئله ۴: برای هر سناریو الگوی مناسب (Strategy، Observer، Factory یا Adapter) را با یک جمله دلیل انتخاب کنید؛ سپس خروجی دو اجرای کد را پیش‌بینی کنید.</div>

a) Save the model to disk every time the validation loss improves.
b) Let the user switch between SGD and Adam optimisers from a config file entry `optimizer: adam`
   without an `if` in the training code. (Two patterns work together here.)
c) Use scikit-learn's `SVR` inside our `RegressionExperiment`.
d) Week 11 adds a folder of 64×64 camera images as a new data source.
e) Predict the `lr` column printed by each run (the data and model are the same):

```python
X = np.array([[1.0], [2.0], [3.0]]); y = np.array([2.0, 4.0, 5.0])
Trainer([ProgressLogger(every=1), StepDecay(step_size=1, gamma=0.5)]).fit(
    GradientDescentRegressor(learning_rate=0.4), X, y, max_epochs=2)
Trainer([StepDecay(step_size=1, gamma=0.5), ProgressLogger(every=1)]).fit(
    GradientDescentRegressor(learning_rate=0.4), X, y, max_epochs=2)
```

Do the two runs train differently, or only print differently?

In [ ]:
import sys
from pathlib import Path

import numpy as np


def find_repo_root(start: Path) -> Path:
    """Walk up from ``start`` until the folder that contains ``src/aerospace_ai``."""
    for folder in (start, *start.parents):
        if (folder / "src" / "aerospace_ai").is_dir():
            return folder
    raise FileNotFoundError("Open this notebook from inside the course repository.")


REPO_ROOT = find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT / "src"))

from aerospace_ai.models import (  # noqa: E402
    GradientDescentRegressor,
    KNearestNeighbors,
    LinearLeastSquares,
    MeanBaseline,
    rmse,
)
from aerospace_ai.training import EarlyStopping, ProgressLogger, StepDecay, Trainer  # noqa: E402

# Scratch space for verifying your hand calculations.